# Lab 03: Causal Multi-Head Self-Attention & LayerNorm from Scratch

**Companion Module:** [Module 05: NLP, CV & Multimodal AI](../../curriculum/04-generative-ai/05_nlp_cv_and_multimodal.md)  
**Difficulty:** Intermediate | **Environment:** Pure Python 3.11+, NumPy

---

## 1. Problem Statement
In autoregressive Decoder Transformers (such as GPT-4, Gemini, and LLaMA), sequence generation must adhere to strict causality:
1. **Future Token Leakage:** During training, token at timestep $t$ must not attend to future tokens at $t+1, t+2$.
2. **Multi-Head Projection:** Projecting tokens into multiple subspace heads allows the model to attend simultaneously to syntactic and semantic relationships.
3. **Layer Normalization:** Stabilizing activations across the hidden dimension to prevent vanishing and exploding gradients.

---

## 2. Learning Objectives
By the end of this lab, you will be able to:
- Construct Multi-Head causal attention in pure NumPy.
- Implement upper-triangular masking to mathematically guarantee zero lookahead bias.
- Implement Layer Normalization and verify token-wise unit variance.

---

## 3. Required Dependencies
```bash
pip install numpy
```


In [ ]:
import numpy as np
print(f"NumPy Version: {np.__version__}")


---

## 4. Dataset Description
We simulate a batched sequence of embedded tokens:
- Shape `(B, T, D)`: $B=1$ batch, $T=4$ tokens, $D=8$ hidden embedding dimension.
- Two attention heads ($H=2$), each with head dimension $d_k = 4$ ($D = H \times d_k$).


In [ ]:
# 5. Implementation Part 1: Causal Multi-Head Attention in Pure NumPy
def multi_head_causal_attention(
    x: np.ndarray,
    w_q: np.ndarray,
    w_k: np.ndarray,
    w_v: np.ndarray,
    w_o: np.ndarray,
    num_heads: int = 2,
) -> np.ndarray:
    """Computes causal multi-head self-attention forward pass in pure NumPy."""
    seq_len, d_model = x.shape
    d_k = d_model // num_heads

    # Project inputs to Query, Key, Value
    q = (x @ w_q).reshape(seq_len, num_heads, d_k).swapaxes(0, 1)  # (H, T, d_k)
    k = (x @ w_k).reshape(seq_len, num_heads, d_k).swapaxes(0, 1)  # (H, T, d_k)
    v = (x @ w_v).reshape(seq_len, num_heads, d_k).swapaxes(0, 1)  # (H, T, d_k)

    # Scaled Dot-Product: (H, T, d_k) @ (H, d_k, T) -> (H, T, T)
    scores = (q @ k.swapaxes(-1, -2)) / np.sqrt(d_k)

    # Upper triangular mask (above main diagonal) set to -inf
    mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    scores = np.where(mask, -1e9, scores)

    # Softmax over sequence length
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    weights = np.exp(shifted) / np.sum(np.exp(shifted), axis=-1, keepdims=True)

    # Weighted sum: (H, T, T) @ (H, T, d_k) -> (H, T, d_k)
    out = (weights @ v).swapaxes(0, 1).reshape(seq_len, d_model)
    return out @ w_o

# Set deterministic random seed
rng = np.random.default_rng(42)
seq_len, d_model = 4, 8

x = rng.normal(size=(seq_len, d_model))
w_q = rng.normal(size=(d_model, d_model)) * 0.1
w_k = rng.normal(size=(d_model, d_model)) * 0.1
w_v = rng.normal(size=(d_model, d_model)) * 0.1
w_o = rng.normal(size=(d_model, d_model)) * 0.1

out1 = multi_head_causal_attention(x, w_q, w_k, w_v, w_o)

# Verify Causal Masking: Mutate future token at index 3 completely
x_mutated = x.copy()
x_mutated[3] *= 500.0

out2 = multi_head_causal_attention(x_mutated, w_q, w_k, w_v, w_o)

# Output for earlier tokens t=0, 1, 2 must be IDENTICAL
assert np.allclose(out1[:3], out2[:3], atol=1e-6), "Future token leaked to past!"
print("Causal Self-Attention Verified: Zero future token leakage.")
print(f"Output Tensor Shape: {out1.shape}")


In [ ]:
# 5. Implementation Part 2: Layer Normalization
def layer_norm(x: np.ndarray, eps: float = 1e-5) -> np.ndarray:
    """Standardizes across the last dimension (d_model)."""
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

normed = layer_norm(out1)
assert np.allclose(normed.mean(axis=-1), 0.0, atol=1e-5)
assert np.allclose(normed.std(axis=-1), 1.0, atol=1e-2)
print("LayerNorm Verified: Mean is 0.0, Std Dev is 1.0.")


---

## 6. Results and Observations
1. **Causal Masking:** Modifying token 3 had zero impact on the outputs of tokens 0, 1, and 2, guaranteeing that generation remains autoregressive.
2. **Scaling Factor:** Dividing dot-product logits by $\sqrt{d_k} = \sqrt{4} = 2.0$ preserved unit variance and prevented softmax saturation.
3. **Layer Normalization:** Normalized activations across the feature dimension independently per token position.

---

## 7. Hands-on Exercises
- **Exercise 1:** Modify `multi_head_causal_attention` to accept 4 attention heads ($H=4, d_k=2$).
- **Exercise 2:** Implement an MLP projection layer with GELU activation following the attention block.

---

## 8. Further Exploration
- Review [Module 05: NLP, Transformers, CV & Multimodal AI](../../curriculum/04-generative-ai/05_nlp_cv_and_multimodal.md).
- Inspect the [Beginner Project: Autograd & Mini-Transformer](../../projects/beginner/README.md).
